In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split

data = pd.read_csv("./temp/processed_data.csv")
train, test = train_test_split(data, test_size=0.2, random_state=42)
train, val = train_test_split(train, test_size=0.25, random_state=42)
print(f"Train Shape: {train.shape} | Val Shape: {val.shape} | Test Shape: {test.shape}")


Train Shape: (296, 170) | Val Shape: (99, 170) | Test Shape: (99, 170)


In [2]:
from pytorch_tabular import TabularModel
from pytorch_tabular.config import (
    DataConfig,
    OptimizerConfig,
    TrainerConfig,
)
from pytorch_tabular.models import GANDALFConfig

from src.pipeline import get_continuous_categorical_columns

# X_train, y_train = split_feature_and_target(train, target_column="target")
# X_val, y_val = split_feature_and_target(val, target_column="target")
# X_test, y_test = split_feature_and_target(test, target_column="target")

cont_cols, cat_cols = get_continuous_categorical_columns()

data_config = DataConfig(
    target=[
        "TARGET"
    ],  # target should always be a list
    continuous_cols=cont_cols,
    categorical_cols=cat_cols,
)
trainer_config = TrainerConfig(
    batch_size=1024,
    max_epochs=100,
)
optimizer_config = OptimizerConfig()
model_config = GANDALFConfig(
    task="classification",
    gflu_stages=6,
    gflu_feature_init_sparsity=0.3,
    gflu_dropout=0.0,
    learning_rate=1e-3,
)

tabular_model = TabularModel(
    data_config=data_config,
    model_config=model_config,
    optimizer_config=optimizer_config,
    trainer_config=trainer_config,
    verbose=True
)

2026-10-03 11:45:53,835 - {pytorch_tabular.tabular_model:145} - INFO - Experiment Tracking is turned off


In [3]:
tabular_model.fit(train=train, validation=val)

Seed set to 42
2026-10-03 11:46:01,959 - {pytorch_tabular.tabular_model:547} - INFO - Preparing the DataLoaders
2026-10-03 11:46:01,972 - {pytorch_tabular.tabular_datamodule:527} - INFO - Setting up the datamodule for classification task
2026-10-03 11:46:02,092 - {pytorch_tabular.tabular_model:598} - INFO - Preparing the Model: GANDALFModel
2026-10-03 11:46:02,153 - {pytorch_tabular.tabular_model:341} - INFO - Preparing the Trainer
GPU available: True (mps), used: True
TPU available: False, using: 0 TPU cores
/Users/minhcnd/repos/comp90051-project/.venv/lib/python3.14/site-packages/pytorch_lightning/trainer/connectors/logger_connector/logger_connector.py:76: Starting from v1.9.0, `tensorboardX` has been removed as a dependency of the `pytorch_lightning` package, due to potential conflicts with other packages in the ML ecosystem. For this reason, `logger=True` will use `CSVLogger` as the default logger, unless the `tensorboard` or `tensorboardX` packages are found. Please `pip install l

┏━━━┳━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━┳━━━━━━━━┳━━━━━━━┳━━━━━━━┓
┃   ┃ Name             ┃ Type             ┃ Params ┃ Mode  ┃ FLOPs ┃
┡━━━╇━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━╇━━━━━━━━╇━━━━━━━╇━━━━━━━┩
│ 0 │ _backbone        │ GANDALFBackbone  │  1.3 M │ train │     0 │
│ 1 │ _embedding_layer │ Embedding1dLayer │    709 │ train │     0 │
│ 2 │ _head            │ Sequential       │    388 │ train │     0 │
│ 3 │ loss             │ CrossEntropyLoss │      0 │ train │     0 │
└───┴──────────────────┴──────────────────┴────────┴───────┴───────┘

Trainable params: 1.3 M                                                                                            
Non-trainable params: 0                                                                                            
Total params: 1.3 M                                                                                                
Total estimated model params size (MB): 5.331                                                                      
Modules in train mode: 117                                                                                         
Modules in eval mode: 0                                                                                            
Total FLOPs: 0

/Users/minhcnd/repos/comp90051-project/.venv/lib/python3.14/site-packages/rich/live.py:260: UserWarning: install 
"ipywidgets" for Jupyter support
  warnings.warn('install "ipywidgets" for Jupyter support')

/Users/minhcnd/repos/comp90051-project/.venv/lib/python3.14/site-packages/pytorch_lightning/utilities/_pytree.py:21
: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` 
instead.

/Users/minhcnd/repos/comp90051-project/.venv/lib/python3.14/site-packages/pytorch_lightning/trainer/connectors/data
_connector.py:434: The 'val_dataloader' does not have many workers which may be a bottleneck. Consider increasing 
the value of the `num_workers` argument` to `num_workers=7` in the `DataLoader` to improve performance.

/Users/minhcnd/repos/comp90051-project/.venv/lib/python3.14/site-packages/torch/utils/data/dataloader.py:759: 
UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)

/Users/minhcnd/repos/comp90051-project/.venv/lib/python3.14/site-packages/pytorch_lightning/trainer/connectors/data
_connector.py:434: The 'train_dataloader' does not have many workers which may be a bottleneck. Consider increasing
the value of the `num_workers` argument` to `num_workers=7` in the `DataLoader` to improve performance.

/Users/minhcnd/repos/comp90051-project/.venv/lib/python3.14/site-packages/pytorch_lightning/loops/fit_loop.py:321: 
The number of training batches (1) is smaller than the logging interval Trainer(log_every_n_steps=50). Set a lower 
value for log_every_n_steps if you want to see logs for the training epoch.

2026-10-03 11:46:09,805 - {pytorch_tabular.tabular_model:690} - INFO - Training the model completed
2026-10-03 11:46:09,805 - {pytorch_tabular.tabular_model:1531} - INFO - Loading the best model


In [4]:
data.head(10).to_csv("data_head_10.csv", index=False)